In [ ]:
import wrds
import pandas as pd
import os

WRDS_FOLDER = "wrdshistoricaldata"
NAMES_FILE = os.path.join(WRDS_FOLDER, "stocknames_lookup.csv")

if not os.path.exists(NAMES_FILE):
    db = wrds.Connection()
    stocknames = db.raw_sql("""
        SELECT permno, ticker, comnam, namedt, nameenddt
        FROM crsp.stocknames
    """, date_cols=['namedt', 'nameenddt'])
    db.close()
    stocknames.to_csv(NAMES_FILE, index=False)
    print(f"Saved {len(stocknames)} rows from crsp.stocknames")
else:
    print(f"Using existing lookup: {NAMES_FILE}")

## Load stuff in

In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Trailing 12-month (minus most recent month) momentum
#
# Signal on date t:
#     price(t - 1 month) / price(t - 12 months) - 1
#
# Uses raw CLOSE prices for strategy stocks.
# SPY is benchmark-only and is excluded from the ranking universe.
# ---------------------------------------------------------------------------

TRADING_DAYS_PER_MONTH = 21

price_1m_ago = prices.shift(TRADING_DAYS_PER_MONTH)
price_12m_ago = prices.shift(TRADING_DAYS_PER_MONTH * 12)

# A signal is valid only if BOTH historical prices exist.
valid_signal = price_1m_ago.notna() & price_12m_ago.notna()

momentum_signal = (price_1m_ago / price_12m_ago) - 1

# Exclude SPY from the strategy.
if "SPY" in momentum_signal.columns:
    momentum_signal["SPY"] = np.nan

# Mask invalid historical observations.
momentum_signal = momentum_signal.where(valid_signal)
# In your Step 2 cell, after momentum_signal is computed as before
# (from the now-unmasked `prices`), add this line before ranking:
momentum_signal = momentum_signal.where(eligible)
print("Momentum signal shape:", momentum_signal.shape)

print(
    "Non-NaN signal values on the most recent date:",
    momentum_signal.iloc[-1].notna().sum()
)

latest_valid_date = momentum_signal.dropna(how="all").index[-1]

top_10 = (
    momentum_signal.loc[latest_valid_date]
    .dropna()
    .sort_values(ascending=False)
    .head(10)
)

print(f"\nTop 10 momentum tickers as of {latest_valid_date}:")
print(top_10)

In [ ]:
# ---------------------------------------------------------------------------
# STEP 3: Pick monthly rebalance dates and rank tickers at each one
# ---------------------------------------------------------------------------

# Force datetime conversion here, explicitly, so this cell doesn't depend
# on remembering to run a fix somewhere earlier.
prices.index = pd.to_datetime(prices.index)
opens.index = pd.to_datetime(opens.index)
momentum_signal.index = pd.to_datetime(momentum_signal.index)

print("prices.index type:", type(prices.index))

month_keys = prices.index.to_period("M")

rebalance_dates = (
    pd.Series(prices.index, index=month_keys)
    .groupby(level=0)
    .first()
    .tolist()
)

first_valid_signal_date = momentum_signal.dropna(how="all").index[0]
rebalance_dates = [d for d in rebalance_dates if d >= first_valid_signal_date]

print("Number of rebalance dates:", len(rebalance_dates))
print("First rebalance date:", rebalance_dates[0])
print("Last rebalance date:", rebalance_dates[-1])


def get_top_decile(rebalance_date, top_pct=0.10, signal=None):
    if signal is None:
        signal = momentum_signal
    signal_today = signal.loc[rebalance_date].dropna()
    if len(signal_today) == 0:
        return []
    n_holdings = max(1, int(len(signal_today) * top_pct))
    top_tickers = signal_today.sort_values(ascending=False).head(n_holdings).index.tolist()
    return top_tickers


for d in rebalance_dates[:2] + rebalance_dates[-2:]:
    picks = get_top_decile(d)
    print(f"{d.date()}: {len(picks)} tickers picked, e.g. {picks[:5]}")

In [ ]:
## STEP 4

assert isinstance(prices.index, pd.DatetimeIndex), "prices.index must be datetime"
assert isinstance(opens.index, pd.DatetimeIndex), "opens.index must be datetime"
assert all(isinstance(d, pd.Timestamp) for d in rebalance_dates[:5]), \
    "rebalance_dates must contain Timestamps"

# CRSP delisting-code convention: 100 = still active, no delisting event.
# Any other non-null code (200s mergers, 400s liquidations, 500s other/
# dropped, etc.) indicates a real delisting.
CRSP_ACTIVE_CODE = 100


def is_real_delisting(dl, rebal_date):
    if dl is None:
        return False
    dlstdt, dlret, dlstcd = dl
    if pd.isna(dlstdt) or dlstdt > rebal_date:
        return False
    if pd.isna(dlstcd) or dlstcd == CRSP_ACTIVE_CODE:
        return False
    return True


def backtest_momentum(rebalance_dates, top_pct=0.10, starting_cash=10000, fee=0.001):
    cash = starting_cash
    holdings = {}
    account_value_log = []
    rebalance_log = []

    for i, rebal_date in enumerate(rebalance_dates):
        picks = get_top_decile(rebal_date, top_pct)

        # --- SELL ---
        for ticker in list(holdings.keys()):
            if ticker not in picks:
                price = opens.loc[rebal_date, ticker]
                if pd.notna(price):
                    shares = holdings.pop(ticker)
                    trade_value = shares * price
                    cash += trade_value * (1 - fee)
                    rebalance_log.append([rebal_date, "SELL", ticker, price, shares])
                else:
                    dl = delisting_info.get(ticker)
                    if is_real_delisting(dl, rebal_date):
                        shares = holdings.pop(ticker)
                        dlstdt, dlret, dlstcd = dl
                        if pd.notna(dlret):
                            last_price = prices[ticker].loc[:rebal_date].dropna()
                            if not last_price.empty:
                                exit_price = last_price.iloc[-1] * (1 + dlret)
                                trade_value = max(exit_price, 0) * shares
                                cash += trade_value * (1 - fee)
                                rebalance_log.append([rebal_date, "SELL_DELISTED", ticker, exit_price, shares])
                            else:
                                rebalance_log.append([rebal_date, "SELL_NO_PRICE_NO_HISTORY", ticker, None, shares])
                        else:
                            # Real delisting, but no return on file - can't
                            # value the exit, log it rather than guessing.
                            rebalance_log.append([rebal_date, "SELL_NO_PRICE_NO_DLRET", ticker, dlstcd, shares])
                    else:
                        # Not a real delisting (either no record, or
                        # dlstcd == 100/active) - just a data gap.
                        # Keep the position, retry next rebalance.
                        rebalance_log.append([rebal_date, "HOLD_MISSING_PRICE", ticker, None, holdings[ticker]])

        # --- BUY ---
        new_picks = [t for t in picks if t not in holdings]
        remaining = len(new_picks)
        for ticker in new_picks:
            if remaining <= 0:
                break
            budget_per_ticker = cash / remaining
            price = opens.loc[rebal_date, ticker]
            remaining -= 1
            if pd.isna(price):
                continue
            shares = (budget_per_ticker // price)
            if shares <= 0:
                continue
            trade_value = shares * price
            cash -= trade_value * (1 + fee)
            holdings[ticker] = shares
            rebalance_log.append([rebal_date, "BUY", ticker, price, shares])

        # --- Mark to market ---
        holdings_value = 0
        for ticker, shares in holdings.items():
            price = prices.loc[rebal_date, ticker]
            if pd.notna(price):
                holdings_value += shares * price
        account_value = cash + holdings_value
        account_value_log.append([rebal_date, account_value])

    account_value_df = pd.DataFrame(account_value_log, columns=["Date", "Account Value"])
    rebalance_df = pd.DataFrame(rebalance_log, columns=["Date", "Action", "Ticker", "Price", "Shares"])

    return account_value_df, rebalance_df


account_value_df, rebalance_df = backtest_momentum(rebalance_dates, top_pct=0.10)

print(account_value_df.head())
print(account_value_df.tail())
print("\nTotal rebalance actions:", len(rebalance_df))
print(rebalance_df["Action"].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import time

total_start = time.time()

WRDS_FOLDER = "wrdshistoricaldata"
YFINANCE_FOLDER = "yfinancehistoricaldata"
MEMBERSHIP_FILE = os.path.join(WRDS_FOLDER, "sp500_membership_post1957.csv")
NAMES_FILE = os.path.join(WRDS_FOLDER, "stocknames_lookup.csv")

# FIX: was hardcoded to 2024-12-31 with an unresolved TODO, while later cells
# in this notebook reported results through 2026-09-11 - those two facts
# can't both be true of the same run. Default to None = use every date your
# SPY file actually has. Set an explicit pd.Timestamp here only if you
# deliberately want a held-out test window, and know that's what you're doing.
BACKTEST_END_DATE = None

membership = pd.read_csv(MEMBERSHIP_FILE, parse_dates=['start', 'ending'])
membership_lookup = {}
for permno, group in membership.groupby('permno'):
    membership_lookup[permno] = list(zip(group['start'], group['ending']))

stocknames = pd.read_csv(NAMES_FILE, parse_dates=['namedt', 'nameenddt'])
best_ticker_lookup = (
    stocknames.sort_values('nameenddt')
    .groupby('permno')['ticker']
    .last()
    .to_dict()
)


def is_member(date, windows):
    return any(start <= date <= end for start, end in windows)


def resolve_column_key(permno, fallback_ticker):
    resolved = best_ticker_lookup.get(permno)
    if resolved and str(resolved).strip() and str(resolved).lower() != 'nan':
        return str(resolved)
    if fallback_ticker and fallback_ticker != "UNK":
        return fallback_ticker
    return f"PERMNO_{permno}"


def load_wrds_universe():
    """
    Returns FOUR objects:
      - prices: RAW unmasked prc_adj, full trading history for every PERMNO.
      - opens:  open_adj = openprc.abs() / cfacpr, same adjustment basis as
                prices. FIX: every ticker that has a `prices` column now
                also gets an `opens` column, even if raw openprc/cfacpr
                were missing from its CSV - filled with NaN instead of
                being silently absent. Previously a missing column meant
                opens.loc[date, ticker] raised KeyError the first time that
                ticker got picked, rather than the NaN your buy/sell logic
                already knows how to handle.
      - eligible: boolean date x ticker matrix, True only when that PERMNO
                  was an actual S&P 500 member on that date.
      - delisting_info: key -> (dlstdt, dlret, dlstcd) for tickers that
                        have a delisting date on file.
    """
    files = glob.glob(os.path.join(WRDS_FOLDER, "PERMNO_*.csv"))

    close_frames = {}
    open_frames = {}
    eligible_frames = {}
    key_to_permnos = {}
    delisting_info = {}

    for f in files:
        df = pd.read_csv(f, parse_dates=['date', 'dlstdt'])
        if df.empty or 'prc_adj' not in df.columns:
            continue

        permno = int(df['permno'].iloc[0])
        old_ticker = str(df['ticker'].iloc[-1])
        key = resolve_column_key(permno, old_ticker)

        df = df.set_index('date').sort_index()
        windows = membership_lookup.get(permno, [])
        eligible_series = pd.Series(
            [is_member(d, windows) for d in df.index],
            index=df.index
        )

        key_to_permnos.setdefault(key, []).append(permno)
        close_frames[key] = df['prc_adj'].abs()
        eligible_frames[key] = eligible_series

        # FIX: always create an opens entry for this key, aligned to this
        # ticker's own date index, so the column exists downstream even
        # when the source data lacks openprc/cfacpr.
        if 'openprc' in df.columns and 'cfacpr' in df.columns:
            open_frames[key] = df['openprc'].abs() / df['cfacpr']
        else:
            open_frames[key] = pd.Series(np.nan, index=df.index)

        # --- delisting capture ---
        dl_rows = df.dropna(subset=['dlstdt'])
        if not dl_rows.empty:
            last_dl = dl_rows.iloc[-1]
            delisting_info[key] = (
                last_dl['dlstdt'],
                last_dl.get('dlret', np.nan),
                last_dl.get('dlstcd', np.nan),
            )

        collisions = {k: p for k, p in key_to_permnos.items() if len(p) > 1}
    if collisions:
        print(f"WARNING: {len(collisions)} column keys map to multiple PERMNOs, "
              f"disambiguating by appending PERMNO to avoid silent data loss.")

    # FIX: instead of overwriting colliding tickers (which silently discards
    # one company's entire history), give each PERMNO its own column key.
    # The common ticker keeps its plain name only when there's no collision;
    # colliding PERMNOs get suffixed so every company's data survives.
    final_close = {}
    final_open = {}
    final_eligible = {}
    final_delisting = {}
    permno_to_final_key = {}

    for key, permnos in key_to_permnos.items():
        if len(permnos) == 1:
            final_close[key] = close_frames[key]
            final_open[key] = open_frames[key]
            final_eligible[key] = eligible_frames[key]
            if key in delisting_info:
                final_delisting[key] = delisting_info[key]
            permno_to_final_key[permnos[0]] = key
        else:
            # NOTE: close_frames/eligible_frames/delisting_info as currently
            # built only retain the LAST-loaded PERMNO's data per key (dict
            # overwrite in the loop above). To truly recover every colliding
            # PERMNO's data, re-run the file loop keyed by permno instead of
            # by ticker - see re-load block below.
            pass

    print(f"Reconstructing {len(collisions)} colliding tickers by PERMNO...")

    # Re-load only the colliding files, this time keyed by permno so nothing
    # gets overwritten.
    collision_permnos = {p for permnos in collisions.values() for p in permnos}
    for f in files:
        df_check = pd.read_csv(f, usecols=['permno'], nrows=1)
        permno = int(df_check['permno'].iloc[0])
        if permno not in collision_permnos:
            continue

        df = pd.read_csv(f, parse_dates=['date', 'dlstdt'])
        if df.empty or 'prc_adj' not in df.columns:
            continue

        old_ticker = str(df['ticker'].iloc[-1])
        base_key = resolve_column_key(permno, old_ticker)
        final_key = f"{base_key}_{permno}"

        df = df.set_index('date').sort_index()
        windows = membership_lookup.get(permno, [])
        eligible_series = pd.Series(
            [is_member(d, windows) for d in df.index], index=df.index
        )

        final_close[final_key] = df['prc_adj'].abs()
        final_eligible[final_key] = eligible_series
        if 'openprc' in df.columns and 'cfacpr' in df.columns:
            final_open[final_key] = df['openprc'].abs() / df['cfacpr']
        else:
            final_open[final_key] = pd.Series(np.nan, index=df.index)

        dl_rows = df.dropna(subset=['dlstdt'])
        if not dl_rows.empty:
            last_dl = dl_rows.iloc[-1]
            final_delisting[final_key] = (
                last_dl['dlstdt'], last_dl.get('dlret', np.nan), last_dl.get('dlstcd', np.nan)
            )
        permno_to_final_key[permno] = final_key

    prices = pd.DataFrame(final_close).sort_index()
    opens = pd.DataFrame(final_open).sort_index()
    eligible = pd.DataFrame(final_eligible).sort_index().fillna(False)
    delisting_info = final_delisting

    return prices, opens, eligible, delisting_info

prices, opens, eligible, delisting_info = load_wrds_universe()

spy_file = f"{YFINANCE_FOLDER}/SPY_adjusted.csv"
spy = pd.read_csv(spy_file)
spy["date"] = pd.to_datetime(spy["date"])
spy = spy.sort_values("date").set_index("date")

# FIX: derive the true end of usable data from WRDS itself, rather than
# hardcoding a date (which goes stale) or trusting SPY's range (which
# runs ahead of WRDS and will silently produce an empty, cash-only tail).
# "Usable" = the last date where at least MIN_COVERAGE tickers have a
# non-null price, so a handful of stray/late rows don't set the cutoff.
MIN_COVERAGE = 50  # tune to whatever "enough of a universe to rank" means to you

coverage_by_date = prices.notna().sum(axis=1)
dates_with_coverage = coverage_by_date[coverage_by_date >= MIN_COVERAGE].index
wrds_max_date = dates_with_coverage.max()

spy_max_date = spy.index.max()

if wrds_max_date < spy_max_date:
    print(
        f"WARNING: WRDS data only supports ranking through {wrds_max_date.date()}, "
        f"but SPY data runs to {spy_max_date.date()}. Capping master_dates at "
        f"{wrds_max_date.date()} so the backtest doesn't silently go to cash "
        f"for the {(spy_max_date - wrds_max_date).days} days in between."
    )

BACKTEST_END_DATE = min(wrds_max_date, spy_max_date)
master_dates = spy.index[spy.index <= BACKTEST_END_DATE]

prices = prices.reindex(master_dates)
opens = opens.reindex(master_dates)
eligible = eligible.reindex(master_dates).fillna(False)

spy_adj = spy["adj_close"].reindex(master_dates)
spy_open = spy["open"].reindex(master_dates)
prices["SPY"] = spy_adj
opens["SPY"] = spy_open
eligible["SPY"] = False

print("prices shape:", prices.shape)
print("eligible shape:", eligible.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("Eligible tickers on final date:", eligible.iloc[-1].sum())
print("Non-null prices on final date:", prices.drop(columns=['SPY']).iloc[-1].notna().sum())
print("Tickers with delisting info captured:", len(delisting_info))
total_elapsed = time.time() - total_start

print(
    f"\nTOTAL RUNTIME: {total_elapsed:.2f} seconds "
    f"({total_elapsed / 60:.2f} minutes)"
)

In [ ]:
# ---------------------------------------------------------------------------
# Quick sanity check: compare momentum's ending value against SPY buy-and-hold
# over the exact same date range.
# ---------------------------------------------------------------------------

spy_start_price = prices.loc[rebalance_dates[0], "SPY"] if "SPY" in prices.columns else None

if spy_start_price is not None:
    spy_end_price = prices.loc[rebalance_dates[-1], "SPY"]
    spy_shares = 10000 / spy_start_price
    spy_ending_value = spy_shares * spy_end_price
    print(f"SPY buy-and-hold over same period: ${spy_ending_value:,.2f}")
    print(f"Momentum strategy: ${account_value_df['Account Value'].iloc[-1]:,.2f}")
else:
    print("SPY not in `prices` columns — was it included in largetickerlist?")

# Also check: did any single rebalance produce a suspiciously huge account
# value jump? That's the signature of a bad split-adjustment or bad price row.
account_value_df["pct_change"] = account_value_df["Account Value"].pct_change()
print("\nLargest single-period jumps:")
print(account_value_df.nlargest(5, "pct_change")[["Date", "Account Value", "pct_change"]])
sndk_trades = rebalance_df[rebalance_df["Ticker"] == "SNDK"]
print(sndk_trades)
sndk_check = pd.read_csv("historicaldata/SNDK.csv")
sndk_check["date"] = pd.to_datetime(sndk_check["date"])
print(sndk_check[(sndk_check["date"] >= "2024-06-01") & (sndk_check["date"] <= "2026-08-01")][["date","open","close"]].to_string())

# Break account value growth into pre-2024 and 2024-onward segments,
# to see how much of the total gain happened in the recent AI/chip rally
# versus being earned steadily across the whole 19-year window.
pre_2024 = account_value_df[account_value_df["Date"] < "2024-01-01"]
post_2024 = account_value_df[account_value_df["Date"] >= "2024-01-01"]

print("Account value end of 2023:", pre_2024["Account Value"].iloc[-1])
print("Account value at start of backtest:", account_value_df["Account Value"].iloc[0])
print("Gain 2007-2023:", pre_2024["Account Value"].iloc[-1] - account_value_df["Account Value"].iloc[0])
print("Gain 2024-2026:", account_value_df["Account Value"].iloc[-1] - pre_2024["Account Value"].iloc[-1])

In [ ]:
# ---------------------------------------------------------------------------
# Walk-forward validation for the momentum strategy
#
# Slices rebalance_dates into rolling multi-year windows, runs a FRESH
# backtest_momentum() on each window independently (starting from $10,000
# each time), and compares against SPY buy-and-hold over that same window.
# ---------------------------------------------------------------------------

def get_walkforward_windows(rebalance_dates, window_years=5, step_years=1):
    """
    Returns a list of (window_start_date, window_end_date, window_rebalance_dates)
    tuples, sliding forward by step_years each time.
    """
    dates = pd.DatetimeIndex(rebalance_dates)
    start = dates.min()
    end = dates.max()

    windows = []
    current_start = start

    while True:
        current_end = current_start + pd.DateOffset(years=window_years)
        if current_end > end:
            break

        window_dates = [d for d in rebalance_dates if current_start <= d < current_end]

        if len(window_dates) > 1:
            windows.append((current_start, current_end, window_dates))

        current_start = current_start + pd.DateOffset(years=step_years)

    return windows

# Old, non adjusted
'''
def spy_return_over_window(window_rebalance_dates):
    """SPY buy-and-hold return over the same dates a momentum window trades."""
    start_price = prices.loc[window_rebalance_dates[0], "SPY"]
    end_price = prices.loc[window_rebalance_dates[-1], "SPY"]
    if pd.isna(start_price) or pd.isna(end_price):
        return None
    return (end_price / start_price - 1) * 100
'''
def spy_return_over_window(window_rebalance_dates):
    """SPY total return including dividends over the same window."""
    
    start_date = pd.Timestamp(window_rebalance_dates[0]).normalize()
    end_date = pd.Timestamp(window_rebalance_dates[-1]).normalize()

    if start_date not in spy_adj.index or end_date not in spy_adj.index:
        return None

    start_price = spy_adj.loc[start_date]
    end_price = spy_adj.loc[end_date]

    if pd.isna(start_price) or pd.isna(end_price):
        return None

    return (end_price / start_price - 1) * 100
    
def backtest_momentum_walkforward(rebalance_dates, window_years=5, step_years=1,
                                    top_pct=0.10, starting_cash=10000, fee=0.001):
    windows = get_walkforward_windows(rebalance_dates, window_years, step_years)
    results = []

    for i, (win_start, win_end, win_dates) in enumerate(windows):
        account_value_df, rebalance_log_df = backtest_momentum(
            win_dates, top_pct=top_pct, starting_cash=starting_cash, fee=fee
        )

        ending_value = account_value_df["Account Value"].iloc[-1]
        momentum_return_pct = (ending_value / starting_cash - 1) * 100

        spy_return_pct = spy_return_over_window(win_dates)

        years_actual = (win_dates[-1] - win_dates[0]).days / 365.25
        momentum_cagr = ((ending_value / starting_cash) ** (1 / years_actual) - 1) * 100 if years_actual > 0 else None
        spy_cagr = (((spy_return_pct / 100) + 1) ** (1 / years_actual) - 1) * 100 if spy_return_pct is not None and years_actual > 0 else None

        results.append({
            "Window": i,
            "Start": win_start.date(),
            "End": win_end.date(),
            "Momentum Return %": round(momentum_return_pct, 1),
            "SPY Return %": round(spy_return_pct, 1) if spy_return_pct is not None else None,
            "Momentum CAGR %": round(momentum_cagr, 2) if momentum_cagr is not None else None,
            "SPY CAGR %": round(spy_cagr, 2) if spy_cagr is not None else None,
            "Beat SPY": momentum_return_pct > spy_return_pct if spy_return_pct is not None else None
        })

    return pd.DataFrame(results)


walkforward_results = backtest_momentum_walkforward(
    rebalance_dates, window_years=5, step_years=1
)

print(walkforward_results.to_string(index=False))

print("\n--- Summary ---")
print("Windows tested:", len(walkforward_results))
print("Windows where momentum beat SPY:", walkforward_results["Beat SPY"].sum(),
      "/", walkforward_results["Beat SPY"].notna().sum())
print("Average momentum CAGR:", round(walkforward_results["Momentum CAGR %"].mean(), 2), "%")
print("Average SPY CAGR:", round(walkforward_results["SPY CAGR %"].mean(), 2), "%")
print("Worst momentum window return:", walkforward_results["Momentum Return %"].min(), "%")
print("Best momentum window return:", walkforward_results["Momentum Return %"].max(), "%")

In [ ]:
def ibkr_commission(shares, price, per_share=0.0035, minimum=0.35, max_pct=0.01,
                     pass_through_multiplier=1.4):
    if shares <= 0:
        return 0.0
    trade_value = shares * price
    commission = max(shares * per_share, minimum)
    commission = min(commission, trade_value * max_pct)
    return commission * pass_through_multiplier


def backtest_momentum_ibkr(rebalance_dates, top_pct=0.10, starting_cash=10000, signal=None):
    cash = starting_cash
    holdings = {}
    account_value_log = []
    rebalance_log = []
    total_commissions = 0.0

    for i, rebal_date in enumerate(rebalance_dates):
        picks = get_top_decile(rebal_date, top_pct, signal=signal)

        # --- SELL ---
        for ticker in list(holdings.keys()):
            if ticker not in picks:
                price = opens.loc[rebal_date, ticker]
                if pd.notna(price):
                    shares = holdings.pop(ticker)
                    trade_value = shares * price
                    commission = ibkr_commission(shares, price)
                    cash += trade_value - commission
                    total_commissions += commission
                    rebalance_log.append([rebal_date, "SELL", ticker, price, shares, commission])
                else:
                    dl = delisting_info.get(ticker)
                    if is_real_delisting(dl, rebal_date):
                        shares = holdings.pop(ticker)
                        dlstdt, dlret, dlstcd = dl
                        if pd.notna(dlret):
                            last_price = prices[ticker].loc[:rebal_date].dropna()
                            if not last_price.empty:
                                exit_price = last_price.iloc[-1] * (1 + dlret)
                                trade_value = max(exit_price, 0) * shares
                                commission = ibkr_commission(shares, exit_price) if exit_price > 0 else 0.0
                                cash += trade_value - commission
                                total_commissions += commission
                                rebalance_log.append([rebal_date, "SELL_DELISTED", ticker, exit_price, shares, commission])
                            else:
                                rebalance_log.append([rebal_date, "SELL_NO_PRICE_NO_HISTORY", ticker, None, shares, 0])
                        else:
                            rebalance_log.append([rebal_date, "SELL_NO_PRICE_NO_DLRET", ticker, dlstcd, shares, 0])
                    else:
                        rebalance_log.append([rebal_date, "HOLD_MISSING_PRICE", ticker, None, holdings[ticker], 0])

        # --- BUY ---
        new_picks = [t for t in picks if t not in holdings]
        remaining = len(new_picks)
        for ticker in new_picks:
            if remaining <= 0:
                break
            budget_per_ticker = cash / remaining
            price = opens.loc[rebal_date, ticker]
            remaining -= 1
            if pd.isna(price):
                continue
            shares = budget_per_ticker // price
            if shares <= 0:
                continue
            trade_value = shares * price
            commission = ibkr_commission(shares, price)
            total_cost = trade_value + commission
            if total_cost > cash:
                continue
            cash -= total_cost
            total_commissions += commission
            holdings[ticker] = shares
            rebalance_log.append([rebal_date, "BUY", ticker, price, shares, commission])

        holdings_value = sum(
            shares * prices.loc[rebal_date, ticker]
            for ticker, shares in holdings.items()
            if pd.notna(prices.loc[rebal_date, ticker])
        )
        account_value = cash + holdings_value
        account_value_log.append([rebal_date, account_value])

    account_value_df = pd.DataFrame(account_value_log, columns=["Date", "Account Value"])
    rebalance_df = pd.DataFrame(rebalance_log, columns=["Date", "Action", "Ticker", "Price", "Shares", "Commission"])

    #print(f"Total commissions paid: ${total_commissions:,.2f}")
    return account_value_df, rebalance_df


account_value_df_ibkr, rebalance_df_ibkr = backtest_momentum_ibkr(rebalance_dates, top_pct=0.10)

print(account_value_df_ibkr.tail())
print(f"\nEnding value with IBKR fees: ${account_value_df_ibkr['Account Value'].iloc[-1]:,.2f}")

In [ ]:
import pandas as pd
from ib_async import *
import matplotlib.pyplot as plt
import asyncio
import time
import numpy as np
import os
from datetime import datetime
# ---------------------------------------------------------------------------
# STEP 5: Compare across rebalance frequencies (daily, weekly, biweekly,
# monthly, quarterly, semiannual, annual)
#
# NOTE on pandas quirks discovered along the way:
# - "A" is deprecated -> use "Y"
# - to_period("3D")/("5D") bucket by a fixed epoch, not by your data's
#   actual trading days, so they silently fail to group trading-day gaps
#   correctly. Use index slicing instead for N-trading-day spacing.
# - to_period("2W")/("2Q") accept the multiplier without error, but each
#   Period's actual span stays 1 week / 1 quarter -- the multiplier is
#   silently ignored. Build base-frequency dates, then take every Nth one.
# ---------------------------------------------------------------------------

def get_rebalance_dates(freq_code):
    """Calendar-anchored rebalance dates (safe for D, W, M, Q, Y base freqs).
    Takes the first trading day within each period bucket."""
    period_keys = prices.index.to_period(freq_code)
    dates = (
        pd.Series(prices.index, index=period_keys)
        .groupby(level=0)
        .first()
        .tolist()
    )
    return [d for d in dates if d >= first_valid_signal_date]


def get_rebalance_dates_every_nth(freq_code, n=1):
    """Build base-frequency rebalance dates, then take every nth one.
    Sidesteps the 2W/2Q multiplier bug -- use n=2 for 'every other', etc."""
    base_dates = get_rebalance_dates(freq_code)
    return base_dates[::n]


def get_rebalance_dates_by_trading_days(n):
    """Every Nth actual trading day, not calendar day -- use this for
    daily or any sub-weekly spacing instead of to_period('ND')."""
    dates = prices.index[prices.index >= first_valid_signal_date]
    return dates[::n].tolist()


REBALANCE_FREQS = {
    "Daily":      lambda: get_rebalance_dates_by_trading_days(1),
    "Weekly":     lambda: get_rebalance_dates("W"),
    "Biweekly":   lambda: get_rebalance_dates_every_nth("W", 2),
    "Monthly":    lambda: get_rebalance_dates("M"),
    "Quarterly":  lambda: get_rebalance_dates("Q"),
    "Semiannual": lambda: get_rebalance_dates_every_nth("Q", 2),
    "Annual":     lambda: get_rebalance_dates("Y"),
}

PERIODS_PER_YEAR = {
    "Daily": 252,
    "Weekly": 52,
    "Biweekly": 26,
    "Monthly": 12,
    "Quarterly": 4,
    "Semiannual": 2,
    "Annual": 1,
}

rebalance_comparison = []

for label, date_fn in REBALANCE_FREQS.items():
    dates = date_fn()
    periods_per_year = PERIODS_PER_YEAR[label]

    acct_df, rebal_df = backtest_momentum_ibkr(dates, top_pct=0.10)

    ending_value = acct_df["Account Value"].iloc[-1]
    total_commissions = rebal_df["Commission"].sum()

    acct_df["Return"] = acct_df["Account Value"].pct_change()
    rets = acct_df["Return"].dropna()
    sharpe = (rets.mean() / rets.std()) * (periods_per_year ** 0.5) if rets.std() > 0 else None

    peak = acct_df["Account Value"].cummax()
    drawdown = (acct_df["Account Value"] - peak) / peak

    rebalance_comparison.append({
        "Rebalance": label,
        "# Rebalances": len(dates),
        "Ending Value": round(ending_value, 2),
        "Total Commissions": round(total_commissions, 2),
        "Commissions % of Ending Value": round(total_commissions / ending_value * 100, 2),
        "Sharpe": round(sharpe, 2) if sharpe is not None else None,
        "Max Drawdown %": round(drawdown.min() * 100, 2),
        "# Trades": len(rebal_df),
    })

rebalance_comparison_df = pd.DataFrame(rebalance_comparison)
print(rebalance_comparison_df.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# STEP 6: Full-History Parameter Heatmaps + CSV EXPORT
#
# Runs 7 rebalance frequencies x 7 top_pct values = 49 backtests over the
# entire available history in your IBKR dataset.
#
# Outputs (saved to plots/):
#   - Raw grid results CSV
#   - One CSV per metric (+ pivot CSV per heatmap)
#   - Four labeled PNG heatmaps
# ---------------------------------------------------------------------------

from pathlib import Path

TOP_PCTS = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30]

PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STEP 6 — FULL-HISTORY PARAMETER SENSITIVITY")
print("=" * 80)
print(f"Output folder: {PLOTS_DIR.resolve()}")
print("Running 49 full-history backtests...\n")

# ---------------------------------------------------------------------------
# RUN ALL 49 BACKTESTS
# ---------------------------------------------------------------------------

grid_results = []

for label, date_fn in REBALANCE_FREQS.items():
    print(f"\n{'-' * 70}\nREBALANCE: {label}\n{'-' * 70}")

    dates = date_fn()
    periods_per_year = PERIODS_PER_YEAR[label]

    for top_pct in TOP_PCTS:
        print(f"  Running {label} / Top {top_pct:.0%}...")

        acct_df, rebal_df = backtest_momentum_ibkr(dates, top_pct=top_pct)

        ending_value = acct_df["Account Value"].iloc[-1]
        total_commissions = rebal_df["Commission"].sum()

        acct_df["Return"] = acct_df["Account Value"].pct_change()
        rets = acct_df["Return"].dropna()
        sharpe = (rets.mean() / rets.std()) * (periods_per_year ** 0.5) if rets.std() > 0 else None

        peak = acct_df["Account Value"].cummax()
        drawdown = (acct_df["Account Value"] - peak) / peak
        max_drawdown = drawdown.min() * 100

        commission_pct = (total_commissions / ending_value * 100) if ending_value != 0 else None

        grid_results.append({
            "Rebalance": label,
            "top_pct": top_pct,
            "Ending Value": ending_value,
            "Sharpe": sharpe,
            "Max Drawdown %": max_drawdown,
            "Commissions % of Ending Value": commission_pct,
        })

grid_df = pd.DataFrame(grid_results)

# ---------------------------------------------------------------------------
# SAVE RAW RESULTS
# ---------------------------------------------------------------------------

raw_csv = PLOTS_DIR / "step6_full_history_grid_results.csv"
grid_df.to_csv(raw_csv, index=False)

print(f"\n{'=' * 80}\n✓ RAW RESULTS SAVED: {raw_csv}\n{'=' * 80}")

# ---------------------------------------------------------------------------
# HEATMAP FUNCTION
# ---------------------------------------------------------------------------

def plot_and_save_heatmap(metric, filename, title, fmt="{:.2f}"):
    """Pivot grid_df on `metric`, draw a heatmap, save PNG + pivot CSV."""

    freq_order = list(REBALANCE_FREQS.keys())

    pivot = grid_df.pivot(index="Rebalance", columns="top_pct", values=metric)
    pivot = pivot.reindex(freq_order).reindex(sorted(pivot.columns), axis=1)

    # Lower-is-better metrics get their color data negated; printed values stay real.
    color_data = -pivot if metric in ("Max Drawdown %", "Commissions % of Ending Value") else pivot

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(color_data.values, cmap="RdYlGn", aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{p:.0%}" for p in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Top Momentum %")
    ax.set_ylabel("Rebalance Frequency")
    ax.set_title(title, fontsize=14, fontweight="bold")

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=9)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metric)

    output_path = PLOTS_DIR / filename
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"✓ HEATMAP SAVED: {output_path}")

    csv_path = PLOTS_DIR / filename.replace(".png", "_data.csv")
    pivot.to_csv(csv_path)
    print(f"✓ HEATMAP DATA SAVED: {csv_path}")

    return pivot

# ---------------------------------------------------------------------------
# GENERATE + SAVE ALL FOUR HEATMAPS
# ---------------------------------------------------------------------------

print(f"\n{'=' * 80}\nGENERATING FULL-HISTORY HEATMAPS\n{'=' * 80}")

ending_value_pivot = plot_and_save_heatmap(
    "Ending Value", "step6_FULL_HISTORY_ending_value_heatmap.png",
    "FULL HISTORY — Ending Value", fmt="{:,.0f}"
)

sharpe_pivot = plot_and_save_heatmap(
    "Sharpe", "step6_FULL_HISTORY_sharpe_heatmap.png",
    "FULL HISTORY — Sharpe Ratio", fmt="{:.2f}"
)

drawdown_pivot = plot_and_save_heatmap(
    "Max Drawdown %", "step6_FULL_HISTORY_max_drawdown_heatmap.png",
    "FULL HISTORY — Maximum Drawdown", fmt="{:.1f}"
)

commission_pivot = plot_and_save_heatmap(
    "Commissions % of Ending Value", "step6_FULL_HISTORY_commissions_heatmap.png",
    "FULL HISTORY — Commissions % of Ending Value", fmt="{:.1f}"
)

# ---------------------------------------------------------------------------
# ALSO SAVE CLEAN TABLE CSVs
# ---------------------------------------------------------------------------

ending_value_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_ending_value.csv")
sharpe_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_sharpe.csv")
drawdown_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_max_drawdown.csv")
commission_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_commissions.csv")

# ---------------------------------------------------------------------------
# PRINT READABLE TABLES
# ---------------------------------------------------------------------------

print(f"\n{'=' * 80}\nFULL-HISTORY RESULTS\n{'=' * 80}")

for metric in ["Ending Value", "Sharpe", "Max Drawdown %", "Commissions % of Ending Value"]:
    print(f"\n--- {metric} ---")
    pivot = grid_df.pivot(index="Rebalance", columns="top_pct", values=metric)
    pivot = pivot.reindex(list(REBALANCE_FREQS.keys()))
    print(pivot.round(0 if metric == "Ending Value" else 2).to_string())

# ---------------------------------------------------------------------------
# FINAL FILE SUMMARY
# ---------------------------------------------------------------------------

print(f"\n{'=' * 80}\nSTEP 6 COMPLETE\n{'=' * 80}")
print("\nFiles saved to plots/:")
for f in sorted(PLOTS_DIR.glob("step6_*")):
    print(f"  {f.name}")

print(f"\n{'=' * 80}\n49 BACKTESTS COMPLETE — ALL RESULTS EXPORTED\n{'=' * 80}")

In [ ]:
# ---------------------------------------------------------------------------
# STEP 6: Full-History Parameter Heatmaps + CSV EXPORT
#
# Runs 7 rebalance frequencies x 7 top_pct values = 49 backtests over the
# entire available history in your IBKR dataset.
#
# Outputs (saved to plots/):
#   - Raw grid results CSV
#   - One CSV per metric (+ pivot CSV per heatmap)
#   - Four labeled PNG heatmaps
# ---------------------------------------------------------------------------

from pathlib import Path

TOP_PCTS = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30]

PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STEP 6 — FULL-HISTORY PARAMETER SENSITIVITY")
print("=" * 80)
print(f"Output folder: {PLOTS_DIR.resolve()}")
print("Running 49 full-history backtests...\n")

# ---------------------------------------------------------------------------
# RUN ALL 49 BACKTESTS
# ---------------------------------------------------------------------------

grid_results = []

for label, date_fn in REBALANCE_FREQS.items():
    print(f"\n{'-' * 70}\nREBALANCE: {label}\n{'-' * 70}")

    dates = date_fn()
    periods_per_year = PERIODS_PER_YEAR[label]

    for top_pct in TOP_PCTS:
        print(f"  Running {label} / Top {top_pct:.0%}...")

        acct_df, rebal_df = backtest_momentum_ibkr(dates, top_pct=top_pct)

        ending_value = acct_df["Account Value"].iloc[-1]
        total_commissions = rebal_df["Commission"].sum()

        acct_df["Return"] = acct_df["Account Value"].pct_change()
        rets = acct_df["Return"].dropna()
        sharpe = (rets.mean() / rets.std()) * (periods_per_year ** 0.5) if rets.std() > 0 else None

        peak = acct_df["Account Value"].cummax()
        drawdown = (acct_df["Account Value"] - peak) / peak
        max_drawdown = drawdown.min() * 100

        commission_pct = (total_commissions / ending_value * 100) if ending_value != 0 else None

        grid_results.append({
            "Rebalance": label,
            "top_pct": top_pct,
            "Ending Value": ending_value,
            "Sharpe": sharpe,
            "Max Drawdown %": max_drawdown,
            "Commissions % of Ending Value": commission_pct,
        })

grid_df = pd.DataFrame(grid_results)

# ---------------------------------------------------------------------------
# SAVE RAW RESULTS
# ---------------------------------------------------------------------------

raw_csv = PLOTS_DIR / "step6_full_history_grid_results.csv"
grid_df.to_csv(raw_csv, index=False)

print(f"\n{'=' * 80}\n✓ RAW RESULTS SAVED: {raw_csv}\n{'=' * 80}")

# ---------------------------------------------------------------------------
# HEATMAP FUNCTION
# ---------------------------------------------------------------------------

def plot_and_save_heatmap(metric, filename, title, fmt="{:.2f}"):
    """Pivot grid_df on `metric`, draw a heatmap, save PNG + pivot CSV."""

    freq_order = list(REBALANCE_FREQS.keys())

    pivot = grid_df.pivot(index="Rebalance", columns="top_pct", values=metric)
    pivot = pivot.reindex(freq_order).reindex(sorted(pivot.columns), axis=1)

    # Lower-is-better metrics get their color data negated; printed values stay real.
    color_data = -pivot if metric in ("Max Drawdown %", "Commissions % of Ending Value") else pivot

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(color_data.values, cmap="RdYlGn", aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{p:.0%}" for p in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Top Momentum %")
    ax.set_ylabel("Rebalance Frequency")
    ax.set_title(title, fontsize=14, fontweight="bold")

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=9)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metric)

    output_path = PLOTS_DIR / filename
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"✓ HEATMAP SAVED: {output_path}")

    csv_path = PLOTS_DIR / filename.replace(".png", "_data.csv")
    pivot.to_csv(csv_path)
    print(f"✓ HEATMAP DATA SAVED: {csv_path}")

    return pivot

# ---------------------------------------------------------------------------
# GENERATE + SAVE ALL FOUR HEATMAPS
# ---------------------------------------------------------------------------

print(f"\n{'=' * 80}\nGENERATING FULL-HISTORY HEATMAPS\n{'=' * 80}")

ending_value_pivot = plot_and_save_heatmap(
    "Ending Value", "step6_FULL_HISTORY_ending_value_heatmap.png",
    "FULL HISTORY — Ending Value", fmt="{:,.0f}"
)

sharpe_pivot = plot_and_save_heatmap(
    "Sharpe", "step6_FULL_HISTORY_sharpe_heatmap.png",
    "FULL HISTORY — Sharpe Ratio", fmt="{:.2f}"
)

drawdown_pivot = plot_and_save_heatmap(
    "Max Drawdown %", "step6_FULL_HISTORY_max_drawdown_heatmap.png",
    "FULL HISTORY — Maximum Drawdown", fmt="{:.1f}"
)

commission_pivot = plot_and_save_heatmap(
    "Commissions % of Ending Value", "step6_FULL_HISTORY_commissions_heatmap.png",
    "FULL HISTORY — Commissions % of Ending Value", fmt="{:.1f}"
)

# ---------------------------------------------------------------------------
# ALSO SAVE CLEAN TABLE CSVs
# ---------------------------------------------------------------------------

ending_value_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_ending_value.csv")
sharpe_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_sharpe.csv")
drawdown_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_max_drawdown.csv")
commission_pivot.to_csv(PLOTS_DIR / "step6_FULL_HISTORY_commissions.csv")

# ---------------------------------------------------------------------------
# PRINT READABLE TABLES
# ---------------------------------------------------------------------------

print(f"\n{'=' * 80}\nFULL-HISTORY RESULTS\n{'=' * 80}")

for metric in ["Ending Value", "Sharpe", "Max Drawdown %", "Commissions % of Ending Value"]:
    print(f"\n--- {metric} ---")
    pivot = grid_df.pivot(index="Rebalance", columns="top_pct", values=metric)
    pivot = pivot.reindex(list(REBALANCE_FREQS.keys()))
    print(pivot.round(0 if metric == "Ending Value" else 2).to_string())

# ---------------------------------------------------------------------------
# FINAL FILE SUMMARY
# ---------------------------------------------------------------------------

print(f"\n{'=' * 80}\nSTEP 6 COMPLETE\n{'=' * 80}")
print("\nFiles saved to plots/:")
for f in sorted(PLOTS_DIR.glob("step6_*")):
    print(f"  {f.name}")

print(f"\n{'=' * 80}\n49 BACKTESTS COMPLETE — ALL RESULTS EXPORTED\n{'=' * 80}")

## Step 8: Long Run

In [ ]:
# ===========================================================================
# STEP 8 (v3: PARALLEL + CHECKPOINTED + RESUMABLE)
# ===========================================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

_required = ["REBALANCE_FREQS", "backtest_momentum_ibkr", "spy_return_over_window",
             "rebalance_dates", "prices"]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        f"STEP 8 requires these to already be defined: {_missing}. "
        "Run STEP 0-5 (and the walk-forward cell, for spy_return_over_window) first."
    )

TOP_PCTS_RANDOM = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30]
CHECKPOINT_DIR = "step8_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

N_JOBS = -1        # -1 = all cores. Set to a fixed number (e.g. 4) if RAM is tight.
CHUNK_SIZE = 500    # tasks per parallel batch, and per checkpoint save


# ===========================================================================
# RANDOM WINDOW GENERATOR (unchanged)
# ===========================================================================

def get_random_windows(monthly_rebalance_dates, n_samples=200, min_years=3,
                        max_years=10, seed=42, min_start_date=None):
    rng = np.random.default_rng(seed)
    dates = pd.DatetimeIndex(monthly_rebalance_dates).sort_values()

    if min_start_date is not None:
        min_start_date = pd.Timestamp(min_start_date)
        dates = dates[dates >= min_start_date]

    dates = pd.DatetimeIndex(dates)
    if len(dates) < 2:
        return []

    min_periods = int(min_years * 12)
    max_periods = int(max_years * 12)

    valid_start_positions = list(range(0, max(0, len(dates) - min_periods + 1)))
    if not valid_start_positions:
        return []

    windows = []
    for _ in range(n_samples):
        start_idx = int(rng.choice(valid_start_positions))
        n_periods = int(rng.integers(min_periods, max_periods + 1))
        end_idx = start_idx + n_periods - 1
        if end_idx >= len(dates):
            continue
        window_dates = dates[start_idx:end_idx + 1]
        if len(window_dates) < 2:
            continue
        windows.append((window_dates[0], window_dates[-1]))

    return windows


# ===========================================================================
# WORKER — runs ONE backtest cell. Must be picklable, keep it self-contained
# ===========================================================================

def _run_single_backtest(frequency, top_pct, window_number, win_start, win_end,
                          win_dates_values, starting_cash):
    win_dates = pd.DatetimeIndex(win_dates_values)

    if len(win_dates) < 2:
        return None

    acct_df, rebal_df = backtest_momentum_ibkr(
        win_dates, top_pct=top_pct, starting_cash=starting_cash
    )
    if acct_df.empty:
        return None

    ending_value = acct_df["Account Value"].iloc[-1]
    if not np.isfinite(ending_value) or ending_value <= 0:
        return None

    years_actual = (win_dates[-1] - win_dates[0]).days / 365.25
    if years_actual <= 0:
        return None

    momentum_cagr = ((ending_value / starting_cash) ** (1 / years_actual) - 1) * 100
    spy_return_pct = spy_return_over_window(win_dates)

    if (spy_return_pct is not None and np.isfinite(spy_return_pct)
            and (spy_return_pct / 100 + 1) > 0):
        spy_cagr = (((spy_return_pct / 100) + 1) ** (1 / years_actual) - 1) * 100
        beat_spy = momentum_cagr > spy_cagr
    else:
        spy_cagr = np.nan
        beat_spy = np.nan

    return {
        "Frequency": frequency, "Top %": top_pct, "Window": window_number,
        "Start": win_start, "End": win_end,
        "Momentum CAGR %": momentum_cagr, "SPY CAGR %": spy_cagr, "Beat SPY": beat_spy
    }


# ===========================================================================
# CHECKPOINT HELPERS
# ===========================================================================

def _checkpoint_path(run_label):
    return os.path.join(CHECKPOINT_DIR, f"{run_label}_results.csv")

def _load_checkpoint(run_label):
    path = _checkpoint_path(run_label)
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["Start", "End"])
        return df
    return pd.DataFrame(columns=["Frequency", "Top %", "Window", "Start", "End",
                                  "Momentum CAGR %", "SPY CAGR %", "Beat SPY"])

def _save_checkpoint(run_label, df):
    df.to_csv(_checkpoint_path(run_label), index=False)


# ===========================================================================
# MAIN DRIVER — parallel, chunked, resumable
# ===========================================================================

def random_window_parameter_sensitivity(monthly_rebalance_dates, min_start_date,
                                         n_samples=200, min_years=3, max_years=10,
                                         starting_cash=10000, seed=42, run_label="",
                                         n_jobs=N_JOBS, chunk_size=CHUNK_SIZE):

    windows = get_random_windows(
        monthly_rebalance_dates, n_samples=n_samples, min_years=min_years,
        max_years=max_years, seed=seed, min_start_date=min_start_date
    )
    unique_windows = len(set(windows))
    print(f"[{run_label}] {len(windows)} windows drawn, {unique_windows} unique "
          f"(window length fixed at {min_years}y, duplicates from a small "
          f"start-position pool are expected)")

    # precompute each frequency's date index once, not once per task
    frequency_dates = {f: pd.DatetimeIndex(fn()).sort_values()
                        for f, fn in REBALANCE_FREQS.items()}

    # build the full task list, with win_dates already sliced
    tasks = []
    for frequency, freq_dates in frequency_dates.items():
        for top_pct in TOP_PCTS_RANDOM:
            for window_number, (win_start, win_end) in enumerate(windows):
                win_dates = freq_dates[(freq_dates >= win_start) & (freq_dates <= win_end)]
                tasks.append((frequency, top_pct, window_number, win_start, win_end,
                              win_dates.values))

    total = len(tasks)

    # resume from checkpoint
    existing = _load_checkpoint(run_label)
    done_keys = set(zip(existing["Frequency"], existing["Top %"], existing["Window"]))
    remaining = [t for t in tasks if (t[0], t[1], t[2]) not in done_keys]

    print(f"[{run_label}] {len(done_keys)}/{total} already checkpointed, "
          f"{len(remaining)} remaining")

    if not remaining:
        return existing

    results_frames = [existing] if not existing.empty else []
    t_start = time.time()
    n_done_this_run = 0

    for chunk_start in range(0, len(remaining), chunk_size):
        chunk = remaining[chunk_start:chunk_start + chunk_size]

        chunk_results = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(_run_single_backtest)(
                frequency, top_pct, window_number, win_start, win_end,
                win_dates_values, starting_cash
            )
            for (frequency, top_pct, window_number, win_start, win_end,
                 win_dates_values) in chunk
        )

        chunk_df = pd.DataFrame([r for r in chunk_results if r is not None])
        if not chunk_df.empty:
            results_frames.append(chunk_df)

        n_done_this_run += len(chunk)
        combined = pd.concat(results_frames, ignore_index=True) if results_frames else pd.DataFrame()
        _save_checkpoint(run_label, combined)  # saved after every chunk

        elapsed = time.time() - t_start
        rate = elapsed / n_done_this_run
        remaining_n = len(remaining) - n_done_this_run
        eta_min = (remaining_n * rate) / 60
        total_done = len(done_keys) + n_done_this_run
        print(f"[{run_label}] {total_done}/{total} done | "
              f"elapsed {elapsed/60:.1f}m | est. remaining {eta_min:.1f}m "
              f"| checkpoint saved")

    final = pd.concat(results_frames, ignore_index=True)
    total_elapsed = time.time() - t_start
    print(f"[{run_label}] DONE in {total_elapsed/60:.1f} minutes "
          f"({total_elapsed/3600:.2f} hours)\n")
    return final


# ===========================================================================
# HEATMAP FUNCTION (unchanged from your fixed v2)
# ===========================================================================

def plot_random_parameter_heatmap(results_df, value_column, title, aggfunc="mean",
                                   fmt=".2f", cmap="RdYlGn"):
    pivot = results_df.pivot_table(
        index="Frequency", columns="Top %", values=value_column, aggfunc=aggfunc
    )
    frequency_order = ["Daily", "Weekly", "Biweekly", "Monthly",
                        "Quarterly", "Semiannual", "Annual"]
    frequency_order = [f for f in frequency_order if f in pivot.index]
    pivot = pivot.reindex(frequency_order)

    plt.figure(figsize=(13, 7))
    plt.imshow(pivot.values, aspect="auto", cmap=cmap)
    plt.colorbar(label=value_column)
    plt.xticks(range(len(pivot.columns)), [f"{x * 100:.0f}%" for x in pivot.columns])
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel("Top Momentum %")
    plt.ylabel("Rebalance Frequency")
    plt.title(title)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            value = pivot.iloc[i, j]
            if pd.notna(value):
                text = f"{value:.1%}" if fmt == ".1%" else f"{value:{fmt}}"
                plt.text(j, i, text, ha="center", va="center")
    plt.tight_layout()
    plt.show()


def format_summary(df):
    out = df.copy()
    out["Top %"] = out["Top %"].map(lambda x: f"{x:.0%}")
    out["Mean_CAGR"] = out["Mean_CAGR"].map(lambda x: f"{x:.2f}%")
    out["Median_CAGR"] = out["Median_CAGR"].map(lambda x: f"{x:.2f}%")
    out["Worst_CAGR"] = out["Worst_CAGR"].map(lambda x: f"{x:.2f}%")
    out["Best_CAGR"] = out["Best_CAGR"].map(lambda x: f"{x:.2f}%")
    out["Win_Rate"] = out["Win_Rate"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "n/a")
    return out


def build_summary(df):
    return (
        df.groupby(["Frequency", "Top %"])
        .agg(
            Mean_CAGR=("Momentum CAGR %", "mean"),
            Median_CAGR=("Momentum CAGR %", "median"),
            Worst_CAGR=("Momentum CAGR %", "min"),
            Best_CAGR=("Momentum CAGR %", "max"),
            Win_Rate=("Beat SPY", "mean"),
            Samples=("Window", "count")
        )
        .reset_index()
    )


# ===========================================================================
# RUN: 15-year and 20-year tests
# ===========================================================================

overall_start = time.time()

random_sensitivity_15y = random_window_parameter_sensitivity(
    rebalance_dates, min_start_date=None,
    n_samples=200, min_years=15, max_years=15, starting_cash=10000, seed=42,
    run_label="15y"
)

random_sensitivity_20y = random_window_parameter_sensitivity(
    rebalance_dates, min_start_date=None,
    n_samples=200, min_years=20, max_years=20, starting_cash=10000, seed=42,
    run_label="20y"
)

print(f"TOTAL RUNTIME BOTH RUNS: {(time.time() - overall_start)/3600:.2f} hours")

# ===========================================================================
# SUMMARY TABLES + HEATMAPS (unchanged logic, reads from the DataFrames above)
# ===========================================================================

summary_15y = build_summary(random_sensitivity_15y)
summary_20y = build_summary(random_sensitivity_20y)

print("\n" + "=" * 80)
print("RECENT 15-YEAR RANDOM-WINDOW PARAMETER SENSITIVITY")
print("=" * 80)
print(format_summary(summary_15y).to_string(index=False))

print("\n" + "=" * 80)
print("RECENT 20-YEAR RANDOM-WINDOW PARAMETER SENSITIVITY")
print("=" * 80)
print(format_summary(summary_20y).to_string(index=False))

for label, df in [("15-Year", random_sensitivity_15y), ("20-Year", random_sensitivity_20y)]:
    print("\n" + "=" * 80)
    print(f"{label.upper()} HEATMAPS")
    print("=" * 80)
    plot_random_parameter_heatmap(df, "Momentum CAGR %", f"{label} Random Windows — Average Momentum CAGR", aggfunc="mean", fmt=".2f")
    plot_random_parameter_heatmap(df, "Momentum CAGR %", f"{label} Random Windows — Median Momentum CAGR", aggfunc="median", fmt=".2f")
    plot_random_parameter_heatmap(df, "Beat SPY", f"{label} Random Windows — Win Rate vs SPY", aggfunc="mean", fmt=".1%")
    plot_random_parameter_heatmap(df, "Momentum CAGR %", f"{label} Random Windows — Worst CAGR", aggfunc="min", fmt=".2f")

print("\n15-YEAR RAW RANDOM-WINDOW RESULTS:")
display(random_sensitivity_15y)

print("\n20-YEAR RAW RANDOM-WINDOW RESULTS:")
display(random_sensitivity_20y)

In [ ]:
# ============================================================
# STEP 8 EXPORT — CSV DATA + LABELED HEATMAPS
# IMPORTANT: THIS CELL DOES NOT RERUN ANY BACKTESTS
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# OUTPUT FOLDER
# ------------------------------------------------------------

PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("EXPORTING RANDOM-WINDOW SENSITIVITY RESULTS")
print("=" * 80)
print(f"Output folder: {PLOTS_DIR.resolve()}")
print("NO BACKTESTS WILL BE RERUN")
print()


# ------------------------------------------------------------
# HELPER: SAVE DATAFRAME TO CSV
# ------------------------------------------------------------

def save_csv(df, filename):
    path = PLOTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"✓ CSV: {path}")


# ------------------------------------------------------------
# HELPER: CREATE + SAVE HEATMAP
# ------------------------------------------------------------

def save_heatmap(
    results_df,
    value_column,
    title,
    filename,
    aggfunc="mean",
    percent=False
):
    # Build pivot
    pivot = results_df.pivot_table(
        index="Frequency",
        columns="Top %",
        values=value_column,
        aggfunc=aggfunc
    )

    # Consistent frequency ordering
    frequency_order = [
        "Daily",
        "Weekly",
        "Biweekly",
        "Monthly",
        "Quarterly",
        "Semiannual",
        "Annual"
    ]

    pivot = pivot.reindex(
        [f for f in frequency_order if f in pivot.index]
    )

    # Consistent column ordering
    pivot = pivot.reindex(
        sorted(pivot.columns),
        axis=1
    )

    # --------------------------------------------------------
    # PLOT
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(13, 7))

    im = ax.imshow(
        pivot.values,
        aspect="auto",
        cmap="RdYlGn"
    )

    cbar = fig.colorbar(im, ax=ax)

    if percent:
        cbar.set_label("Win Rate")
    else:
        cbar.set_label(value_column)

    # X labels
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(
        [f"{x * 100:.0f}%" for x in pivot.columns]
    )

    # Y labels
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    ax.set_xlabel("Top Momentum %")
    ax.set_ylabel("Rebalance Frequency")
    ax.set_title(title, fontsize=14, fontweight="bold")

    # --------------------------------------------------------
    # VALUES INSIDE CELLS
    # --------------------------------------------------------

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):

            value = pivot.iloc[i, j]

            if pd.notna(value):

                if percent:
                    text = f"{value:.1%}"
                else:
                    text = f"{value:.2f}"

                ax.text(
                    j,
                    i,
                    text,
                    ha="center",
                    va="center",
                    fontsize=10
                )

    plt.tight_layout()

    path = PLOTS_DIR / filename

    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"✓ Heatmap: {path}")

    return pivot


# ============================================================
# 15-YEAR EXPORTS
# ============================================================

print("\n" + "-" * 80)
print("15-YEAR RESULTS")
print("-" * 80)

# Raw results
save_csv(
    random_sensitivity_15y,
    "random_sensitivity_15y_raw.csv"
)

# Summary
save_csv(
    summary_15y,
    "random_sensitivity_15y_summary.csv"
)

# ------------------------------------------------------------
# 15Y — AVERAGE CAGR
# ------------------------------------------------------------

pivot_15y_avg = save_heatmap(
    random_sensitivity_15y,
    "Momentum CAGR %",
    "15-Year Random Windows — Average Momentum CAGR",
    "heatmap_15y_average_cagr.png",
    aggfunc="mean"
)

# ------------------------------------------------------------
# 15Y — MEDIAN CAGR
# ------------------------------------------------------------

pivot_15y_median = save_heatmap(
    random_sensitivity_15y,
    "Momentum CAGR %",
    "15-Year Random Windows — Median Momentum CAGR",
    "heatmap_15y_median_cagr.png",
    aggfunc="median"
)

# ------------------------------------------------------------
# 15Y — WIN RATE VS SPY
# ------------------------------------------------------------

win_rate_15y = (
    random_sensitivity_15y
    .groupby(["Frequency", "Top %"])["Beat SPY"]
    .mean()
    .reset_index()
)

save_csv(
    win_rate_15y,
    "random_sensitivity_15y_win_rate.csv"
)

pivot_15y_win = save_heatmap(
    win_rate_15y,
    "Beat SPY",
    "15-Year Random Windows — Win Rate vs SPY",
    "heatmap_15y_win_rate.png",
    aggfunc="mean",
    percent=True
)

# ------------------------------------------------------------
# 15Y — WORST CAGR
# IMPORTANT: MINIMUM, NOT AVERAGE
# ------------------------------------------------------------

pivot_15y_worst = save_heatmap(
    random_sensitivity_15y,
    "Momentum CAGR %",
    "15-Year Random Windows — WORST CAGR",
    "heatmap_15y_worst_cagr.png",
    aggfunc="min"
)


# ============================================================
# 20-YEAR EXPORTS
# ============================================================

print("\n" + "-" * 80)
print("20-YEAR RESULTS")
print("-" * 80)

# Raw results
save_csv(
    random_sensitivity_20y,
    "random_sensitivity_20y_raw.csv"
)

# Summary
save_csv(
    summary_20y,
    "random_sensitivity_20y_summary.csv"
)

# ------------------------------------------------------------
# 20Y — AVERAGE CAGR
# ------------------------------------------------------------

pivot_20y_avg = save_heatmap(
    random_sensitivity_20y,
    "Momentum CAGR %",
    "20-Year Random Windows — Average Momentum CAGR",
    "heatmap_20y_average_cagr.png",
    aggfunc="mean"
)

# ------------------------------------------------------------
# 20Y — MEDIAN CAGR
# ------------------------------------------------------------

pivot_20y_median = save_heatmap(
    random_sensitivity_20y,
    "Momentum CAGR %",
    "20-Year Random Windows — Median Momentum CAGR",
    "heatmap_20y_median_cagr.png",
    aggfunc="median"
)

# ------------------------------------------------------------
# 20Y — WIN RATE VS SPY
# ------------------------------------------------------------

win_rate_20y = (
    random_sensitivity_20y
    .groupby(["Frequency", "Top %"])["Beat SPY"]
    .mean()
    .reset_index()
)

save_csv(
    win_rate_20y,
    "random_sensitivity_20y_win_rate.csv"
)

pivot_20y_win = save_heatmap(
    win_rate_20y,
    "Beat SPY",
    "20-Year Random Windows — Win Rate vs SPY",
    "heatmap_20y_win_rate.png",
    aggfunc="mean",
    percent=True
)

# ------------------------------------------------------------
# 20Y — WORST CAGR
# IMPORTANT: MINIMUM, NOT AVERAGE
# ------------------------------------------------------------

pivot_20y_worst = save_heatmap(
    random_sensitivity_20y,
    "Momentum CAGR %",
    "20-Year Random Windows — WORST CAGR",
    "heatmap_20y_worst_cagr.png",
    aggfunc="min"
)


# ============================================================
# ALSO EXPORT THE HEATMAP MATRICES AS CSV
# ============================================================

print("\n" + "-" * 80)
print("EXPORTING HEATMAP MATRICES")
print("-" * 80)

pivot_15y_avg.to_csv(
    PLOTS_DIR / "heatmap_15y_average_cagr_data.csv"
)

pivot_15y_median.to_csv(
    PLOTS_DIR / "heatmap_15y_median_cagr_data.csv"
)

pivot_15y_win.to_csv(
    PLOTS_DIR / "heatmap_15y_win_rate_data.csv"
)

pivot_15y_worst.to_csv(
    PLOTS_DIR / "heatmap_15y_worst_cagr_data.csv"
)

pivot_20y_avg.to_csv(
    PLOTS_DIR / "heatmap_20y_average_cagr_data.csv"
)

pivot_20y_median.to_csv(
    PLOTS_DIR / "heatmap_20y_median_cagr_data.csv"
)

pivot_20y_win.to_csv(
    PLOTS_DIR / "heatmap_20y_win_rate_data.csv"
)

pivot_20y_worst.to_csv(
    PLOTS_DIR / "heatmap_20y_worst_cagr_data.csv"
)

print("✓ All heatmap matrices exported as CSV")


# ============================================================
# FINAL FILE LIST
# ============================================================

print("\n" + "=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)

files = sorted(PLOTS_DIR.iterdir())

for f in files:
    print(f"  {f.name}")

print("\n" + "=" * 80)
print("DONE — NO BACKTESTS WERE RERUN")
print("=" * 80)

In [ ]:
# ============================================================
# COMMISSION-SENSITIVITY CELL
# Reruns 4 targeted combos across the windows Step 8 already drew,
# under several commission multipliers. Does NOT touch the 9,800-row grid.
# ============================================================
import functools, time
import numpy as np
import pandas as pd

COMBOS_TO_TEST = [("Biweekly", 0.02), ("Biweekly", 0.03), ("Monthly", 0.02), ("Monthly", 0.03)]
MULTIPLIERS_TO_TEST = [1.0, 1.4, 2.0, 2.5, 3.0]
MAX_WINDOWS = 40          # raise once you trust the shape; ~1.5s/backtest at 15y
STARTING_CASH = 10_000

unique_windows = (
    random_sensitivity_15y[["Window", "Start", "End"]]
    .drop_duplicates(subset=["Start", "End"])
    .head(MAX_WINDOWS)
)
print(f"{len(COMBOS_TO_TEST)} combos x {len(unique_windows)} windows x "
      f"{len(MULTIPLIERS_TO_TEST)} multipliers = "
      f"{len(COMBOS_TO_TEST)*len(unique_windows)*len(MULTIPLIERS_TO_TEST)} backtests")

_original_ibkr_commission = ibkr_commission  # real reference, survives the monkeypatch

def _run_at_multiplier(frequency, top_pct, win_start, win_end, multiplier):
    global ibkr_commission
    freq_dates = pd.DatetimeIndex(REBALANCE_FREQS[frequency]()).sort_values()
    win_dates = freq_dates[(freq_dates >= win_start) & (freq_dates <= win_end)]
    if len(win_dates) < 2:
        return None
    ibkr_commission = functools.partial(_original_ibkr_commission, pass_through_multiplier=multiplier)
    try:
        acct_df, rebal_df = backtest_momentum_ibkr(win_dates, top_pct=top_pct, starting_cash=STARTING_CASH)
    finally:
        ibkr_commission = _original_ibkr_commission
    if acct_df.empty:
        return None
    ending_value = acct_df["Account Value"].iloc[-1]
    years = (win_dates[-1] - win_dates[0]).days / 365.25
    if years <= 0 or not np.isfinite(ending_value) or ending_value <= 0:
        return None
    cagr = ((ending_value / STARTING_CASH) ** (1 / years) - 1) * 100
    total_commission = rebal_df["Commission"].sum() if "Commission" in rebal_df else np.nan
    return {"Frequency": frequency, "Top %": top_pct, "Multiplier": multiplier,
            "Start": win_start, "End": win_end, "CAGR %": cagr,
            "Total Commission $": total_commission,
            "Commission % of Ending Value": (total_commission / ending_value * 100) if pd.notna(total_commission) else np.nan}

t0 = time.time()
rows = [_run_at_multiplier(f, tp, w.Start, w.End, m)
        for f, tp in COMBOS_TO_TEST
        for w in unique_windows.itertuples()
        for m in MULTIPLIERS_TO_TEST]
commission_sensitivity = pd.DataFrame([r for r in rows if r is not None])
print(f"Done in {(time.time()-t0)/60:.1f} min")

summary = (commission_sensitivity.groupby(["Frequency", "Top %", "Multiplier"])["CAGR %"]
           .mean().reset_index()
           .pivot_table(index=["Frequency", "Top %"], columns="Multiplier", values="CAGR %"))
print(summary)
commission_sensitivity.to_csv("plots/commission_sensitivity_raw.csv", index=False)
summary.to_csv("plots/commission_sensitivity_summary.csv")

In [ ]:
# ============================================================
# CLIFF-QUANTIFICATION CELL
# Measures Win Rate / CAGR swings between adjacent Top % values
# per frequency. Large swings = fragile result, not a stable edge.
# ============================================================
import pandas as pd

def build_summary_from_raw(raw_df):
    return (raw_df.groupby(["Frequency", "Top %"])
            .agg(Mean_CAGR=("Momentum CAGR %", "mean"),
                 Win_Rate=("Beat SPY", "mean"),
                 Samples=("Window", "count"))
            .reset_index().sort_values(["Frequency", "Top %"]))

def cliff_report(summary_df, label):
    rows = []
    for frequency, group in summary_df.groupby("Frequency"):
        group = group.sort_values("Top %")
        win_diff = group["Win_Rate"].diff().abs()
        cagr_diff = group["Mean_CAGR"].diff().abs()
        rows.append({"Horizon": label, "Frequency": frequency,
                      "Max Win-Rate Jump (pp)": win_diff.max() * 100,
                      "Jump at Top %": group.loc[win_diff.idxmax(), "Top %"] if win_diff.notna().any() else None,
                      "Max CAGR Jump (pp)": cagr_diff.max()})
    return pd.DataFrame(rows).sort_values("Max Win-Rate Jump (pp)", ascending=False)

cliff_combined = pd.concat([
    cliff_report(build_summary_from_raw(random_sensitivity_15y), "15y"),
    cliff_report(build_summary_from_raw(random_sensitivity_20y), "20y"),
], ignore_index=True)
print(cliff_combined.to_string(index=False))

FLAG_THRESHOLD_PP = 15
print(f"\nFlagged (jump > {FLAG_THRESHOLD_PP}pp):")
print(cliff_combined[cliff_combined["Max Win-Rate Jump (pp)"] > FLAG_THRESHOLD_PP].to_string(index=False))
cliff_combined.to_csv("plots/cliff_quantification.csv", index=False)

In [ ]:
# ============================================================
# CONCENTRATION-RISK CELL
# Checks whether Biweekly's edge over Monthly comes from broader
# selection or from repeatedly catching the same few mega-caps.
# ============================================================
import pandas as pd
from collections import Counter

COMBOS_TO_CHECK = [("Biweekly", 0.02), ("Monthly", 0.02), ("Biweekly", 0.03), ("Monthly", 0.03)]
rows = []

for frequency, top_pct in COMBOS_TO_CHECK:
    counter = Counter()
    for d in REBALANCE_FREQS[frequency]():
        counter.update(get_top_decile(d, top_pct))
    total_picks = sum(counter.values())
    top5 = counter.most_common(5)
    top5_share = sum(c for _, c in top5) / total_picks if total_picks else float("nan")
    shares = pd.Series(counter.values(), dtype=float) / total_picks
    hhi = (shares ** 2).sum()
    rows.append({"Frequency": frequency, "Top %": top_pct,
                 "Unique Tickers Ever Picked": len(counter),
                 "Total Pick-Events": total_picks,
                 "Top 5 Tickers": ", ".join(t for t, _ in top5),
                 "Top 5 Share of Picks": top5_share,
                 "HHI (higher = more concentrated)": hhi})

concentration_df = pd.DataFrame(rows)
print(concentration_df.to_string(index=False))
concentration_df.to_csv("plots/concentration_risk_summary.csv", index=False)

In [ ]:
# ============================================================
# EFFECTIVE-SAMPLE-SIZE / BLOCK BOOTSTRAP CELL
# Collapses to one row per UNIQUE window (the true independent unit)
# before computing a confidence interval on Win Rate.
# ============================================================
import numpy as np
import pandas as pd

N_BOOTSTRAP = 2000
RNG = np.random.default_rng(0)

def block_bootstrap_win_rate(raw_df, frequency, top_pct):
    sub = raw_df[(raw_df["Frequency"] == frequency) & (raw_df["Top %"] == top_pct)]
    per_window = sub.groupby(["Start", "End"])["Beat SPY"].mean().values.astype(float)
    n_unique = len(per_window)
    boot = np.array([RNG.choice(per_window, size=n_unique, replace=True).mean()
                      for _ in range(N_BOOTSTRAP)])
    return {"Frequency": frequency, "Top %": top_pct,
            "Unique Windows (effective N)": n_unique,
            "Naive Win Rate (raw draws)": sub["Beat SPY"].mean(),
            "Cluster Win Rate": per_window.mean(),
            "Bootstrap 5th pct": np.percentile(boot, 5),
            "Bootstrap 95th pct": np.percentile(boot, 95)}

COMBOS_TO_CHECK = [("Biweekly", 0.02), ("Biweekly", 0.03), ("Monthly", 0.02), ("Monthly", 0.03), ("Annual", 0.02)]
bootstrap_df = pd.DataFrame([
    block_bootstrap_win_rate(df, f, tp)
    for f, tp in COMBOS_TO_CHECK
    for df in (random_sensitivity_15y, random_sensitivity_20y)
])
print(bootstrap_df.to_string(index=False))
bootstrap_df.to_csv("plots/effective_sample_size_bootstrap.csv", index=False)

In [ ]:
print(prices.index.min(), prices.index.max(), prices.shape)
# should read: 1993-01-29 ... 2024-12-31, (8038, 1937) — from your original Step 8 log.
# if it doesn't match, the kernel state is contaminated. Restart and rerun cells 0-14 fresh.

check = pd.merge(
    commission_sensitivity[commission_sensitivity["Multiplier"] == 1.4][["Frequency","Top %","Start","End","CAGR %"]],
    random_sensitivity_15y[["Frequency","Top %","Start","End","Momentum CAGR %"]],
    on=["Frequency","Top %","Start","End"], how="inner"
)
check["diff"] = check["CAGR %"] - check["Momentum CAGR %"]
print(check.sort_values("diff", ascending=False).head(10).to_string(index=False))

In [ ]:
print(type(delisting_info), len(delisting_info))
sample_ticker = "LEH"  # or any name you know delisted pre-2010
print(delisting_info.get(sample_ticker))

# does is_real_delisting still behave as expected on it?
import inspect
print(inspect.getsource(is_real_delisting))

In [ ]:
print(first_valid_signal_date)
print(len(REBALANCE_FREQS["Biweekly"]()))
print(summary_15y[(summary_15y["Frequency"]=="Biweekly") & (summary_15y["Top %"]==0.02)])
print(summary_15y[(summary_15y["Frequency"]=="Monthly") & (summary_15y["Top %"]==0.02)])

In [ ]:
# ============================================================
# SHARED SELL LOGIC
# Factored out of backtest_momentum_ibkr so the three BUY variants
# below don't each duplicate the delisting-handling block.
# ============================================================

def _process_sells(rebal_date, picks, holdings):
    """Sells everything in `holdings` not in `picks`. Mutates holdings.
    Returns (cash_delta, commission_total, log_rows)."""
    cash_delta = 0.0
    commission_total = 0.0
    log_rows = []
    for ticker in list(holdings.keys()):
        if ticker in picks:
            continue
        price = opens.loc[rebal_date, ticker]
        if pd.notna(price):
            shares = holdings.pop(ticker)
            trade_value = shares * price
            commission = ibkr_commission(shares, price)
            cash_delta += trade_value - commission
            commission_total += commission
            log_rows.append([rebal_date, "SELL", ticker, price, shares, commission])
        else:
            dl = delisting_info.get(ticker)
            if is_real_delisting(dl, rebal_date):
                shares = holdings.pop(ticker)
                dlstdt, dlret, dlstcd = dl
                if pd.notna(dlret):
                    last_price = prices[ticker].loc[:rebal_date].dropna()
                    if not last_price.empty:
                        exit_price = last_price.iloc[-1] * (1 + dlret)
                        trade_value = max(exit_price, 0) * shares
                        commission = ibkr_commission(shares, exit_price) if exit_price > 0 else 0.0
                        cash_delta += trade_value - commission
                        commission_total += commission
                        log_rows.append([rebal_date, "SELL_DELISTED", ticker, exit_price, shares, commission])
                    else:
                        log_rows.append([rebal_date, "SELL_NO_PRICE_NO_HISTORY", ticker, None, shares, 0])
                else:
                    log_rows.append([rebal_date, "SELL_NO_PRICE_NO_DLRET", ticker, dlstcd, shares, 0])
            else:
                log_rows.append([rebal_date, "HOLD_MISSING_PRICE", ticker, None, holdings[ticker], 0])
    return cash_delta, commission_total, log_rows


# ============================================================
# BUY VARIANT 1: reproduces the notebook's existing logic exactly
# (sequential cash/remaining split, floor to whole shares). Included
# so it can run through the same harness as the other two, for a fair
# comparison instead of comparing against numbers from a different code path.
# ============================================================

def _buy_whole_sequential(rebal_date, new_picks, cash):
    log_rows, commission_total, new_holdings = [], 0.0, {}
    remaining = len(new_picks)
    for ticker in new_picks:
        if remaining <= 0:
            break
        budget_per_ticker = cash / remaining
        price = opens.loc[rebal_date, ticker]
        remaining -= 1
        if pd.isna(price):
            continue
        shares = budget_per_ticker // price
        if shares <= 0:
            continue
        trade_value = shares * price
        commission = ibkr_commission(shares, price)
        total_cost = trade_value + commission
        if total_cost > cash:
            continue
        cash -= total_cost
        commission_total += commission
        new_holdings[ticker] = shares
        log_rows.append([rebal_date, "BUY", ticker, price, shares, commission])
    return cash, commission_total, log_rows, new_holdings


# ============================================================
# BUY VARIANT 2: fractional shares
# Equal cash split across new picks, no share-count rounding.
# One correction pass for commission since it depends on share count.
# ============================================================

def _buy_fractional(rebal_date, new_picks, cash):
    log_rows, commission_total, new_holdings = [], 0.0, {}
    n = len(new_picks)
    if n == 0:
        return cash, commission_total, log_rows, new_holdings
    budget_per_ticker = cash / n
    for ticker in new_picks:
        price = opens.loc[rebal_date, ticker]
        if pd.isna(price) or price <= 0:
            continue
        shares = budget_per_ticker / price
        commission = ibkr_commission(shares, price)
        shares = max((budget_per_ticker - commission) / price, 0)
        if shares <= 0:
            continue
        trade_value = shares * price
        commission = ibkr_commission(shares, price)
        total_cost = trade_value + commission
        if total_cost > cash:
            continue
        cash -= total_cost
        commission_total += commission
        new_holdings[ticker] = shares
        log_rows.append([rebal_date, "BUY", ticker, price, shares, commission])
    return cash, commission_total, log_rows, new_holdings


# ============================================================
# BUY VARIANT 3: whole shares, greedy cash sweep
# Floor-allocate first, then repeatedly buy one more whole share of the
# cheapest still-affordable pick until nothing fits. This is the part
# the current code doesn't do: it only rolls unspent budget FORWARD
# through the pick list, it never comes back and mops up what's left
# after the last ticker.
# ============================================================

def _buy_whole_greedy(rebal_date, new_picks, cash):
    log_rows, commission_total, new_holdings = [], 0.0, {}
    n = len(new_picks)
    if n == 0:
        return cash, commission_total, log_rows, new_holdings

    prices_now = {}
    for ticker in new_picks:
        p = opens.loc[rebal_date, ticker]
        if pd.notna(p) and p > 0:
            prices_now[ticker] = p
    if not prices_now:
        return cash, commission_total, log_rows, new_holdings

    budget_per_ticker = cash / n
    target_shares = {t: int(budget_per_ticker // p) for t, p in prices_now.items()}
    target_shares = {t: s for t, s in target_shares.items() if s > 0}

    spent = sum(s * prices_now[t] for t, s in target_shares.items())
    leftover = cash - spent

    cheapest_first = sorted(prices_now.items(), key=lambda kv: kv[1])
    bought_more = True
    while bought_more:
        bought_more = False
        for ticker, price in cheapest_first:
            if leftover >= price:
                target_shares[ticker] = target_shares.get(ticker, 0) + 1
                leftover -= price
                bought_more = True

    for ticker, shares in target_shares.items():
        if shares <= 0:
            continue
        price = prices_now[ticker]
        commission = ibkr_commission(shares, price)
        trade_value = shares * price
        total_cost = trade_value + commission
        # commission wasn't accounted for during the sweep, so trim a
        # share at a time in the rare case it pushed a trade over cash
        while shares > 0 and total_cost > cash:
            shares -= 1
            trade_value = shares * price
            commission = ibkr_commission(shares, price) if shares > 0 else 0.0
            total_cost = trade_value + commission
        if shares <= 0:
            continue
        cash -= total_cost
        commission_total += commission
        new_holdings[ticker] = shares
        log_rows.append([rebal_date, "BUY", ticker, price, shares, commission])
    return cash, commission_total, log_rows, new_holdings


# ============================================================
# SHARED DRIVER: identical to backtest_momentum_ibkr's loop
# structure, but takes the buy strategy as a parameter
# ============================================================

def backtest_momentum_sizing(rebalance_dates, buy_fn, top_pct=0.10,
                              starting_cash=10000, signal=None):
    cash = starting_cash
    holdings = {}
    account_value_log = []
    rebalance_log = []
    total_commissions = 0.0

    for rebal_date in rebalance_dates:
        picks = get_top_decile(rebal_date, top_pct, signal=signal)

        cash_delta, sell_commission, sell_rows = _process_sells(rebal_date, picks, holdings)
        cash += cash_delta
        total_commissions += sell_commission
        rebalance_log.extend(sell_rows)

        new_picks = [t for t in picks if t not in holdings]
        cash, buy_commission, buy_rows, new_holdings = buy_fn(rebal_date, new_picks, cash)
        holdings.update(new_holdings)
        total_commissions += buy_commission
        rebalance_log.extend(buy_rows)

        holdings_value = sum(
            shares * prices.loc[rebal_date, ticker]
            for ticker, shares in holdings.items()
            if pd.notna(prices.loc[rebal_date, ticker])
        )
        account_value_log.append([rebal_date, cash + holdings_value])

    account_value_df = pd.DataFrame(account_value_log, columns=["Date", "Account Value"])
    rebalance_df = pd.DataFrame(rebalance_log, columns=["Date", "Action", "Ticker", "Price", "Shares", "Commission"])
    return account_value_df, rebalance_df, total_commissions

In [ ]:
# ============================================================
# COMPARISON CELL
#
# Runs all three sizing methods over the same rebalance dates
# and compares them against dividend-adjusted SPY buy & hold.
#
# Additional metrics:
#   - Ending Value
#   - CAGR
#   - Total Commissions
#   - # Buy Trades
#   - Max Drawdown
#   - Annualized Volatility
#   - Sharpe Ratio
#   - Excess CAGR vs SPY
# ============================================================

import yfinance as yf


# ============================================================
# SETTINGS
# ============================================================

SIZING_MODES = {
    "Whole, sequential (current)": _buy_whole_sequential,
    "Whole, greedy (no idle cash)": _buy_whole_greedy,
    "Fractional": _buy_fractional,
}

TOP_PCT_FOR_COMPARISON = 0.02
STARTING_CASH = 10000

# Risk-free rate used for Sharpe.
# Set to 0 if you want a simple excess-return-free Sharpe.
ANNUAL_RISK_FREE_RATE = 0.00


# ============================================================
# HELPER: PERFORMANCE METRICS
# ============================================================

def calculate_performance_metrics(
    account_df,
    starting_value=10000,
    risk_free_rate=0.00
):
    """
    Calculate performance statistics from an account-value series.

    Returns:
        ending_value
        CAGR
        max_drawdown
        annualized_volatility
        Sharpe ratio
    """

    df = account_df.copy()

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    values = df["Account Value"].astype(float)

    ending_value = values.iloc[-1]

    # --------------------------------------------------------
    # CAGR
    # --------------------------------------------------------

    years = (
        df["Date"].iloc[-1] - df["Date"].iloc[0]
    ).days / 365.25

    if years > 0 and starting_value > 0 and ending_value > 0:
        cagr = (
            (ending_value / starting_value) ** (1 / years) - 1
        )
    else:
        cagr = np.nan

    # --------------------------------------------------------
    # Returns
    #
    # Account values are only recorded at rebalance dates,
    # so calculate returns between consecutive observations.
    # --------------------------------------------------------

    returns = values.pct_change().dropna()

    # Approximate observations per year from actual dates.
    if len(df) > 1:
        date_deltas = (
            df["Date"]
            .diff()
            .dt.days
            .dropna()
        )

        avg_days = date_deltas.mean()

        if avg_days > 0:
            periods_per_year = 365.25 / avg_days
        else:
            periods_per_year = np.nan
    else:
        periods_per_year = np.nan

    # --------------------------------------------------------
    # Annualized Volatility
    # --------------------------------------------------------

    if len(returns) > 1 and periods_per_year > 0:
        annualized_volatility = (
            returns.std(ddof=1)
            * np.sqrt(periods_per_year)
        )
    else:
        annualized_volatility = np.nan

    # --------------------------------------------------------
    # Sharpe Ratio
    #
    # Convert annual risk-free rate to an approximate
    # per-observation rate.
    # --------------------------------------------------------

    if (
        len(returns) > 1
        and periods_per_year > 0
        and annualized_volatility > 0
    ):

        periodic_rf = (
            (1 + risk_free_rate)
            ** (1 / periods_per_year)
            - 1
        )

        excess_returns = returns - periodic_rf

        sharpe = (
            excess_returns.mean()
            / excess_returns.std(ddof=1)
        ) * np.sqrt(periods_per_year)

    else:
        sharpe = np.nan

    # --------------------------------------------------------
    # Maximum Drawdown
    # --------------------------------------------------------

    running_peak = values.cummax()

    drawdown = (
        values / running_peak
    ) - 1

    max_drawdown = drawdown.min()

    return {
        "Ending Value": ending_value,
        "CAGR %": cagr * 100,
        "Max Drawdown %": max_drawdown * 100,
        "Annualized Volatility %": annualized_volatility * 100,
        "Sharpe Ratio": sharpe,
    }


# ============================================================
# RUN THREE MOMENTUM SIZING STRATEGIES
# ============================================================

comparison_dates = REBALANCE_FREQS["Biweekly"]()

sizing_results = {}
sizing_comparison = []


for label, buy_fn in SIZING_MODES.items():

    acct_df, rebal_df, total_comm = backtest_momentum_sizing(
        comparison_dates,
        buy_fn,
        top_pct=TOP_PCT_FOR_COMPARISON,
        starting_cash=STARTING_CASH
    )

    sizing_results[label] = acct_df

    metrics = calculate_performance_metrics(
        acct_df,
        starting_value=STARTING_CASH,
        risk_free_rate=ANNUAL_RISK_FREE_RATE
    )

    sizing_comparison.append({
        "Sizing": label,
        "Ending Value": round(
            metrics["Ending Value"], 2
        ),
        "CAGR %": round(
            metrics["CAGR %"], 2
        ),
        "Max Drawdown %": round(
            metrics["Max Drawdown %"], 2
        ),
        "Annualized Volatility %": round(
            metrics["Annualized Volatility %"], 2
        ),
        "Sharpe Ratio": round(
            metrics["Sharpe Ratio"], 2
        ),
        "Total Commissions": round(
            total_comm, 2
        ),
        "# Buy Trades": int(
            (rebal_df["Action"] == "BUY").sum()
        ),
    })


# ============================================================
# SPY BUY & HOLD — DIVIDENDS INCLUDED
#
# Yahoo Finance Adjusted Close is used for the benchmark.
# This represents SPY total-return growth including
# distributions.
# ============================================================

SPY_TICKER = "SPY"

spy_start_date = comparison_dates[0]
spy_end_date = comparison_dates[-1]


spy_data = yf.download(
    SPY_TICKER,
    start=pd.Timestamp(
        spy_start_date
    ).strftime("%Y-%m-%d"),

    end=(
        pd.Timestamp(spy_end_date)
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d"),

    auto_adjust=False,
    progress=False
)


# Handle yfinance MultiIndex columns.
if isinstance(spy_data.columns, pd.MultiIndex):

    if SPY_TICKER in spy_data.columns.get_level_values(-1):

        spy_adj_close = (
            spy_data["Adj Close"][SPY_TICKER]
        )

    else:

        spy_adj_close = (
            spy_data["Adj Close"].iloc[:, 0]
        )

else:

    spy_adj_close = spy_data["Adj Close"]


spy_adj_close.index = pd.to_datetime(
    spy_adj_close.index
)

spy_adj_close = (
    spy_adj_close
    .sort_index()
    .dropna()
)


# ============================================================
# SPY ENTRY
#
# Use the first available adjusted SPY observation on or
# after the first comparison date.
# ============================================================

comparison_dates_pd = pd.DatetimeIndex(
    pd.to_datetime(comparison_dates)
)

spy_start_idx = spy_adj_close.index.searchsorted(
    comparison_dates_pd[0],
    side="left"
)

if spy_start_idx >= len(spy_adj_close):

    raise ValueError(
        "No SPY adjusted-price data available "
        "for the comparison period."
    )


spy_entry_date = spy_adj_close.index[spy_start_idx]

spy_entry_price = float(
    spy_adj_close.iloc[spy_start_idx]
)


# ============================================================
# BUY SPY ONCE AND HOLD
# ============================================================

spy_shares = (
    STARTING_CASH / spy_entry_price
)


spy_values = []

for date in comparison_dates_pd:

    eligible = spy_adj_close.loc[
        spy_adj_close.index <= date
    ]

    if eligible.empty:

        spy_values.append(np.nan)

    else:

        latest_adj_price = float(
            eligible.iloc[-1]
        )

        spy_values.append(
            spy_shares * latest_adj_price
        )


spy_account_df = pd.DataFrame({
    "Date": comparison_dates,
    "Account Value": spy_values
})


spy_account_df = (
    spy_account_df
    .dropna(subset=["Account Value"])
    .reset_index(drop=True)
)


sizing_results[
    "SPY Buy & Hold (Dividends)"
] = spy_account_df


# ============================================================
# SPY METRICS
# ============================================================

spy_metrics = calculate_performance_metrics(
    spy_account_df,
    starting_value=STARTING_CASH,
    risk_free_rate=ANNUAL_RISK_FREE_RATE
)


sizing_comparison.append({
    "Sizing": "SPY Buy & Hold (Dividends)",
    "Ending Value": round(
        spy_metrics["Ending Value"], 2
    ),
    "CAGR %": round(
        spy_metrics["CAGR %"], 2
    ),
    "Max Drawdown %": round(
        spy_metrics["Max Drawdown %"], 2
    ),
    "Annualized Volatility %": round(
        spy_metrics["Annualized Volatility %"], 2
    ),
    "Sharpe Ratio": round(
        spy_metrics["Sharpe Ratio"], 2
    ),
    "Total Commissions": 0.00,
    "# Buy Trades": 1,
})


# ============================================================
# CREATE COMPARISON TABLE
# ============================================================

sizing_comparison_df = pd.DataFrame(
    sizing_comparison
)


# ============================================================
# ADD EXCESS CAGR VS SPY
#
# This is simply:
#
# Strategy CAGR - SPY CAGR
#
# Positive means the strategy's CAGR was higher than SPY
# during this backtest period.
# ============================================================

spy_cagr = spy_metrics["CAGR %"]

sizing_comparison_df[
    "Excess CAGR vs SPY (pp)"
] = (
    sizing_comparison_df["CAGR %"]
    - spy_cagr
).round(2)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print(
    sizing_comparison_df.to_string(index=False)
)


# ============================================================
# ACCOUNT VALUE CHART
# ============================================================

plt.figure(figsize=(10, 6))

for label, acct_df in sizing_results.items():

    plt.plot(
        acct_df["Date"],
        acct_df["Account Value"],
        label=label
    )

plt.legend()

plt.title(
    f"Account Value: Momentum Sizing vs "
    f"SPY Buy & Hold (Dividends) "
    f"(Top {TOP_PCT_FOR_COMPARISON:.0%})"
)

plt.ylabel("Account Value ($)")
plt.xlabel("Date")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# EX-SNDK ISOLATION TEST
#
# Question: does the strategy still beat SPY once the single
# stock behind most of the 2024-2026 gain (SNDK) is removed
# from the eligible universe entirely? If removing one ticker
# flips a 19-year backtest, "beats SPY" isn't a systematic edge.
# ============================================================

TOP_PCT_CHECK = TOP_PCT_FOR_COMPARISON if "TOP_PCT_FOR_COMPARISON" in globals() else 0.10

assert "SNDK" in momentum_signal.columns, "SNDK not found — check ticker key / PERMNO collision suffix"

momentum_signal_ex_sndk = momentum_signal.copy()
momentum_signal_ex_sndk["SNDK"] = np.nan  # can never be picked, rest of universe untouched

acct_baseline, rebal_baseline = backtest_momentum_ibkr(
    rebalance_dates, top_pct=TOP_PCT_CHECK, starting_cash=STARTING_CASH
)
acct_ex_sndk, rebal_ex_sndk = backtest_momentum_ibkr(
    rebalance_dates, top_pct=TOP_PCT_CHECK, starting_cash=STARTING_CASH,
    signal=momentum_signal_ex_sndk
)

metrics_baseline = calculate_performance_metrics(acct_baseline, starting_value=STARTING_CASH, risk_free_rate=ANNUAL_RISK_FREE_RATE)
metrics_ex_sndk = calculate_performance_metrics(acct_ex_sndk, starting_value=STARTING_CASH, risk_free_rate=ANNUAL_RISK_FREE_RATE)

sndk_comparison_df = pd.DataFrame([
    {"Version": "With SNDK", **metrics_baseline,
     "Excess CAGR vs SPY (pp)": round(metrics_baseline["CAGR %"] - spy_metrics["CAGR %"], 2)},
    {"Version": "Ex-SNDK", **metrics_ex_sndk,
     "Excess CAGR vs SPY (pp)": round(metrics_ex_sndk["CAGR %"] - spy_metrics["CAGR %"], 2)},
])
print(sndk_comparison_df.to_string(index=False))

gain_with = metrics_baseline["Ending Value"] - STARTING_CASH
gain_without = metrics_ex_sndk["Ending Value"] - STARTING_CASH
print(f"\nTotal gain with SNDK eligible:   ${gain_with:,.2f}")
print(f"Total gain with SNDK excluded:   ${gain_without:,.2f}")
print(f"Gain attributable to SNDK alone: ${gain_with - gain_without:,.2f} "
      f"({(gain_with - gain_without) / gain_with:.1%} of total)")